# Smart Waste Collection Simulation
# Data Preparation
#
# Objective:
# Transform the audited smart-bin dataset into modelling-ready
# daily transition data while applying the decisions documented
# during the Data Audit.

In [ ]:
import pandas as pd

df = pd.read_csv(
    "smart_bin_processed.csv",
    parse_dates=["timestamp"]
)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nUnique bins:", df["serialNumber"].nunique())
print("Exact duplicates:", df.duplicated().sum())

print("\nAmbiguous records:")
print(df["ambiguous_bin_date"].value_counts())

Create the modelling-safe working dataset

In [ ]:
# Create a separate working dataset for behaviour modelling
model_df = df[
    (df["ambiguous_bin_date"] == False) &
    (df["serialNumber"] != 1511202)
].copy()

print("Original processed records:", len(df))
print("Modelling records:", len(model_df))
print("Records excluded:", len(df) - len(model_df))

print("\nUnique bins in original dataset:",
      df["serialNumber"].nunique())

print("Unique bins in modelling dataset:",
      model_df["serialNumber"].nunique())

print("\nBin 1511202 records remaining:",
      (model_df["serialNumber"] == 1511202).sum())

print("Ambiguous records remaining:",
      model_df["ambiguous_bin_date"].sum())

Build consecutive-observation variables

In [ ]:
# Sort each bin chronologically
model_df = model_df.sort_values(
    ["serialNumber", "timestamp"]
).copy()

# Previous observation for the same bin
model_df["previous_date"] = (
    model_df
    .groupby("serialNumber")["timestamp"]
    .shift(1)
)

model_df["previous_fullness"] = (
    model_df
    .groupby("serialNumber")["latestFullness"]
    .shift(1)
)

# Number of days between observations
model_df["days_between"] = (
    model_df["timestamp"] - model_df["previous_date"]
).dt.days

print("Days between consecutive observations:")
print(
    model_df["days_between"]
    .value_counts()
    .sort_index()
)

print("\nMissing previous observations:",
      model_df["previous_date"].isna().sum())

print("\nMinimum days between:",
      model_df["days_between"].min())

print("Maximum days between:",
      model_df["days_between"].max())

Create the daily-transition dataset

In [ ]:
# Keep only genuine consecutive-day observations
daily_df = model_df[
    model_df["days_between"] == 1
].copy()

# Calculate daily change in fullness
daily_df["fullness_change"] = (
    daily_df["latestFullness"] -
    daily_df["previous_fullness"]
)

print("Total modelling observations:", len(model_df))
print("Valid one-day transitions:", len(daily_df))

print("\nDaily fullness change distribution:")
print(
    daily_df["fullness_change"]
    .value_counts()
    .sort_index()
)

print("\nMinimum daily change:",
      daily_df["fullness_change"].min())

print("Maximum daily change:",
      daily_df["fullness_change"].max())

Classify the daily transitions

In [ ]:
# Classify each daily transition

def classify_transition(change):
    if change > 0:
        return "Increase"
    elif change < 0:
        return "Decrease"
    else:
        return "No Change"

daily_df["transition_type"] = (
    daily_df["fullness_change"]
    .apply(classify_transition)
)

# Summary
transition_summary = (
    daily_df["transition_type"]
    .value_counts()
    .to_frame("Count")
)

transition_summary["Percentage"] = (
    transition_summary["Count"] / len(daily_df) * 100
).round(2)

print(transition_summary)

print("\nTotal transitions:",
      transition_summary["Count"].sum())

Identify potential emptying/reset events

In [ ]:
# Flag potential emptying/reset events
daily_df["potential_emptying"] = (
    (daily_df["fullness_change"] < 0) &
    (daily_df["latestFullness"] == 0)
)

print("Potential emptying/reset events:")
print(daily_df["potential_emptying"].value_counts())

potential_count = daily_df["potential_emptying"].sum()

print("\nNumber of potential emptying events:",
      potential_count)

print(
    "Percentage of all daily transitions:",
    round(potential_count / len(daily_df) * 100, 2),
    "%"
)

# Check how many decreases do NOT end at zero
other_decreases = daily_df[
    (daily_df["fullness_change"] < 0) &
    (daily_df["latestFullness"] != 0)
]

print("\nOther decreases:", len(other_decreases))

print("\nPotential emptying transitions:")
print(
    daily_df[
        daily_df["potential_emptying"]
    ][
        ["previous_fullness", "latestFullness"]
    ]
    .value_counts()
    .sort_index()
)

Investigate the 222 other decreases

In [ ]:
# Examine decreases that do NOT end at zero
other_decreases = daily_df[
    (daily_df["fullness_change"] < 0) &
    (daily_df["latestFullness"] != 0)
].copy()

print("Total other decreases:", len(other_decreases))

print("\nTransition patterns:")
print(
    other_decreases[
        ["previous_fullness", "latestFullness"]
    ]
    .value_counts()
    .sort_index()
)

print("\nCurrent reason:")
print(
    other_decreases["reason"]
    .value_counts()
)

print("\nNumber of bins involved:",
      other_decreases["serialNumber"].nunique())

print("\nOther decreases by bin:")
print(
    other_decreases["serialNumber"]
    .value_counts()
    .sort_values(ascending=False)
    .to_string()
)

Create the final modelling behaviour category

In [ ]:
# Create final behaviour classification

def classify_behaviour(row):
    if row["fullness_change"] > 0:
        return "Increase"

    elif row["fullness_change"] == 0:
        return "No Change"

    elif row["potential_emptying"]:
        return "Potential Emptying"

    else:
        return "Other Decrease"


daily_df["behaviour_type"] = daily_df.apply(
    classify_behaviour,
    axis=1
)

# Summary table
behaviour_summary = (
    daily_df["behaviour_type"]
    .value_counts()
    .to_frame("Count")
)

behaviour_summary["Percentage"] = (
    behaviour_summary["Count"]
    / len(daily_df) * 100
).round(2)

print(behaviour_summary)

print("\nTotal:", behaviour_summary["Count"].sum())

Time between potential emptying events

In [ ]:
# Create potential emptying events dataset
emptying_df = daily_df[
    daily_df["potential_emptying"]
].copy()

# Sort chronologically within each bin
emptying_df = emptying_df.sort_values(
    ["serialNumber", "timestamp"]
).copy()

# Previous potential emptying date for the same bin
emptying_df["previous_emptying_date"] = (
    emptying_df
    .groupby("serialNumber")["timestamp"]
    .shift(1)
)

# Calendar days between potential emptying events
emptying_df["days_since_previous_emptying"] = (
    emptying_df["timestamp"] -
    emptying_df["previous_emptying_date"]
).dt.days

print("Potential emptying events:",
      len(emptying_df))

print("Bins with potential emptying events:",
      emptying_df["serialNumber"].nunique())

print("\nDays between potential emptying events:")
print(
    emptying_df["days_since_previous_emptying"]
    .describe()
)

print("\nFirst 10 interval counts:")
print(
    emptying_df["days_since_previous_emptying"]
    .value_counts()
    .sort_index()
    .head(10)
)

Investigate the unusually long intervals

In [ ]:
# Inspect unusually long intervals between potential emptying events

print("Intervals greater than 30 days:",
      (emptying_df["days_since_previous_emptying"] > 30).sum())

print("Intervals greater than 60 days:",
      (emptying_df["days_since_previous_emptying"] > 60).sum())

print("Intervals greater than 100 days:",
      (emptying_df["days_since_previous_emptying"] > 100).sum())

print("\nLongest 20 intervals:")

print(
    emptying_df[
        [
            "serialNumber",
            "description",
            "previous_emptying_date",
            "timestamp",
            "days_since_previous_emptying"
        ]
    ]
    .sort_values(
        "days_since_previous_emptying",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)

Check coverage between potential emptying events

In [ ]:
# Calculate observation coverage between potential emptying events

def calculate_interval_coverage(row):

    # First emptying event has no previous event
    if pd.isna(row["previous_emptying_date"]):
        return pd.Series(
            [None, None, None],
            index=[
                "observed_days_between_events",
                "expected_days_between_events",
                "coverage_percent"
            ]
        )

    bin_records = model_df[
        (model_df["serialNumber"] == row["serialNumber"]) &
        (model_df["timestamp"] > row["previous_emptying_date"]) &
        (model_df["timestamp"] <= row["timestamp"])
    ]

    observed = bin_records["timestamp"].nunique()
    expected = int(row["days_since_previous_emptying"])

    coverage = (
        observed / expected * 100
        if expected > 0 else None
    )

    return pd.Series(
        [observed, expected, coverage],
        index=[
            "observed_days_between_events",
            "expected_days_between_events",
            "coverage_percent"
        ]
    )


coverage_results = emptying_df.apply(
    calculate_interval_coverage,
    axis=1
)

emptying_df = pd.concat(
    [emptying_df, coverage_results],
    axis=1
)

print("Coverage summary:")
print(
    emptying_df["coverage_percent"]
    .describe()
)

print("\nIntervals with 100% coverage:",
      (emptying_df["coverage_percent"] == 100).sum())

print("Intervals with less than 100% coverage:",
      (emptying_df["coverage_percent"] < 100).sum())

print("\nLongest intervals with coverage:")
print(
    emptying_df[
        [
            "serialNumber",
            "previous_emptying_date",
            "timestamp",
            "days_since_previous_emptying",
            "observed_days_between_events",
            "expected_days_between_events",
            "coverage_percent"
        ]
    ]
    .sort_values(
        "days_since_previous_emptying",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)

Create reliable emptying intervals

In [ ]:
# Keep only intervals with complete observation coverage
reliable_emptying_intervals = emptying_df[
    emptying_df["coverage_percent"] == 100
].copy()

print("All measurable emptying intervals:",
      emptying_df["days_since_previous_emptying"].notna().sum())

print("Reliable intervals with 100% coverage:",
      len(reliable_emptying_intervals))

print(
    "Percentage retained:",
    round(
        len(reliable_emptying_intervals) /
        emptying_df["days_since_previous_emptying"].notna().sum()
        * 100,
        2
    ),
    "%"
)

print("\nReliable interval statistics:")
print(
    reliable_emptying_intervals[
        "days_since_previous_emptying"
    ].describe()
)

print("\nInterval frequency:")
print(
    reliable_emptying_intervals[
        "days_since_previous_emptying"
    ]
    .value_counts()
    .sort_index()
    .head(20)
)

Check whether daily behaviour differs by current fullness level

In [ ]:
# Behaviour by starting (previous-day) fullness level

behaviour_by_fullness = pd.crosstab(
    daily_df["previous_fullness"],
    daily_df["behaviour_type"]
)

behaviour_by_fullness_pct = pd.crosstab(
    daily_df["previous_fullness"],
    daily_df["behaviour_type"],
    normalize="index"
) * 100

print("Behaviour counts by previous fullness:")
print(behaviour_by_fullness)

print("\nBehaviour percentages by previous fullness:")
print(
    behaviour_by_fullness_pct.round(2)
)

Build the transition probability matrix

In [ ]:
# Count transitions between fullness states
transition_matrix_count = pd.crosstab(
    daily_df["previous_fullness"],
    daily_df["latestFullness"]
)

# Convert counts to row percentages
transition_matrix_pct = pd.crosstab(
    daily_df["previous_fullness"],
    daily_df["latestFullness"],
    normalize="index"
) * 100

print("Transition count matrix:")
print(transition_matrix_count)

print("\nTransition probability matrix (%):")
print(
    transition_matrix_pct.round(2)
)

# Check that every row sums to 100%
print("\nRow totals:")
print(
    transition_matrix_pct.sum(axis=1).round(2)
)

Check threshold representation

In [ ]:
# Check how much each threshold contributes to daily transitions

threshold_summary = (
    daily_df
    .groupby("fullnessThreshold")
    .agg(
        transitions=("serialNumber", "size"),
        bins=("serialNumber", "nunique")
    )
)

threshold_summary["Percentage"] = (
    threshold_summary["transitions"]
    / len(daily_df) * 100
).round(2)

print("Threshold representation:")
print(threshold_summary)

print("\nBehaviour by threshold (%):")
print(
    pd.crosstab(
        daily_df["fullnessThreshold"],
        daily_df["behaviour_type"],
        normalize="index"
    ).mul(100).round(2)
)

Export the prepared files

In [ ]:
# ---------------------------------------------------------
# Export prepared datasets
# ---------------------------------------------------------

# 1. Daily transition dataset
daily_export_columns = [
    "serialNumber",
    "description",
    "timestamp",
    "previous_date",
    "previous_fullness",
    "latestFullness",
    "fullness_change",
    "transition_type",
    "behaviour_type",
    "potential_emptying",
    "reason",
    "fullnessThreshold"
]

daily_df[daily_export_columns].to_csv(
    "smart_bin_daily_transitions.csv",
    index=False
)


# 2. Potential emptying events
emptying_export_columns = [
    "serialNumber",
    "description",
    "timestamp",
    "previous_fullness",
    "latestFullness",
    "previous_emptying_date",
    "days_since_previous_emptying",
    "observed_days_between_events",
    "expected_days_between_events",
    "coverage_percent",
    "reason",
    "fullnessThreshold"
]

emptying_df[emptying_export_columns].to_csv(
    "smart_bin_emptying_events.csv",
    index=False
)


# 3. Historical transition probability matrix
transition_probability_export = (
    transition_matrix_pct
    .round(4)
    .reset_index()
)

transition_probability_export.to_csv(
    "smart_bin_transition_probabilities.csv",
    index=False
)


print("Files exported successfully!")

print("\n1. smart_bin_daily_transitions.csv")
print("Rows:", len(daily_df))

print("\n2. smart_bin_emptying_events.csv")
print("Rows:", len(emptying_df))

print("\n3. smart_bin_transition_probabilities.csv")
print("Rows:", len(transition_probability_export))